In [ ]:
import os
from dotenv import load_dotenv
import openai
import requests
import re

load_dotenv()

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"
messages = [] 

In [ ]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()

def get_movie_details(id: int):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()

def get_movie_credits(id: int):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()

In [ ]:
PROMPT = """
You are a movie expert agent.

You can use the following functions:

1. get_popular_movies()
2. get_movie_details(id)
3. get_movie_credits(id)

IMPORTANT:
- Only respond with the function call.
- Do NOT explain.
- Do NOT add extra text.

Examples:
get_popular_movies()
get_movie_details(550)
get_movie_credits(550)
"""

In [ ]:
tools={
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits      
}

def parse_tool_call(text):
    text = text.strip()

    match = re.fullmatch(r"([a-zA-Z_]\w*)\((.*?)\)", text)
    if not match:
        raise ValueError("Invalid function format")

    func_name = match.group(1)
    args = match.group(2)

    if func_name not in tools:
        raise ValueError("Function not allowed")

    if args == "":
        return func_name, []

    return func_name, [int(args)]
    

In [ ]:
from annotated_types import T


def run_movie_agent():
    user_input = input("Enter your query: ")
    messages = [
        {"role": "system", "content": PROMPT},
        {"role": "user", "content": user_input}
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )

    function_call_text = response.choices[0].message.content.strip()
    print("LLM chose:", function_call_text)

    func_name, args = parse_tool_call(function_call_text)
    result = tools[func_name](*args)
    
    print("\n📦 API Result:")
    print(result)
        



In [ ]:
run_movie_agent()

In [ ]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

messages = []  # 대화 기록 저장

In [ ]:
SYSTEM_PROMPT = """
너는 사용자의 취향을 기억하는 영화 추천 챗봇이야.

규칙:
- 대화 기록을 기반으로 사용자의 취향을 파악해.
- 사용자가 이미 본 영화는 절대 다시 추천하지 마.
- 추천 요청 시 2~3개만 제안해.
- 사용자가 "내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?"
  라고 물으면 이전 대화를 기반으로 정확히 말해.
- 한국어로 답해.
"""

def chat(user_text: str):
    messages.append({"role": "user", "content": user_text})

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, *messages],
        temperature=0.6
    )

    answer = response.choices[0].message.content.strip()

    messages.append({"role": "assistant", "content": answer})

    return answer